In [ ]:
# 🪙 Solana Memecoin Scanner — Google Colab Version (Final Solana-only + Sorted UpTrend)

# === Install dan Import Library ===
!pip install requests pandas matplotlib numpy --quiet

import requests, json, time, sys, pandas as pd, io, base64
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML
from requests.utils import requote_uri

# === CONFIG ===
MAX_TOKENS = 100
DELAY = 0.35
PUBLIC_PROXIES = [
    "https://r.jina.ai/",
    "https://r.jina.ai/http://",
    "https://api.allorigins.win/raw?url=",
    "https://thingproxy.freeboard.io/fetch/",
    "https://corsproxy.io/?"
]
ENDPOINTS = {
    "gmgn": "https://gmgn.ai/defi/quotation/v1/trending?chain=solana",
    "dex": "https://api.dexscreener.com/latest/dex/trending",
    "dex_search": "https://api.dexscreener.com/latest/dex/search?q=",
    "gecko": "https://api.geckoterminal.com/api/v2/networks/solana/trending_pools",
    "pump": "https://frontend-api.pump.fun/trending?chain=sol"
}
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}


# ---------- fetch helper ----------
def fetch_url_text(url, timeout=12):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    for proxy in PUBLIC_PROXIES:
        try:
            proxied = proxy + url
            r = requests.get(proxied, headers=HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.text
        except Exception:
            continue
    return None


def fetch_json(url, timeout=12):
    txt = fetch_url_text(url, timeout=timeout)
    if not txt:
        return None
    try:
        return json.loads(txt)
    except Exception:
        return None


# ---------- dexscreener search ----------
def get_price_and_change_from_dexscreener_by_address(addr):
    if not addr:
        return None
    url = ENDPOINTS["dex_search"] + requote_uri(addr)
    j = fetch_json(url)
    if not j:
        return None
    pairs = j.get("pairs") or []
    for p in pairs:
        chainId = str(p.get("chainId") or "").lower()
        if "solana" in chainId:
            price = p.get("priceUsd") or p.get("price")
            liq = p.get("liquidity", {}).get("usd") if isinstance(p.get("liquidity"), dict) else p.get("liquidity")
            vol = p.get("volume", {}).get("h24") if isinstance(p.get("volume"), dict) else p.get("volume")
            change = None
            for k in ("priceChange", "priceChange24h", "priceChange1h", "price_change_24h"):
                if p.get(k) is not None:
                    change = p.get(k)
                    break
            return {"price": price, "liquidity": liq, "volume": vol, "price_change": change}
    return None


# ---------- helpers ----------
def extract_percent_change_from_obj(obj):
    if obj is None:
        return None
    try:
        if isinstance(obj, (int, float)):
            return float(obj)
        if isinstance(obj, str):
            s = obj.replace("%", "").replace(",", "").strip()
            return float(s)
    except:
        pass
    if isinstance(obj, dict):
        for k in ("percent", "h24", "24h", "change", "change24h"):
            if obj.get(k) is not None:
                return extract_percent_change_from_obj(obj.get(k))
    return None


def render_sparkline(values, figsize=(2.2, 0.5), dpi=100):
    try:
        arr = np.array(values, dtype=float)
        plt.figure(figsize=figsize, dpi=dpi)
        ax = plt.gca()
        ax.plot(arr, linewidth=1.1, color="#007BFF")
        ax.fill_between(range(len(arr)), arr, arr.min(), color="#007BFF", alpha=0.1)
        ax.axis('off')
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
        plt.close()
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('ascii')
        return f'<img src="data:image/png;base64,{img_b64}" style="height:28px;"/>'
    except Exception:
        return ""

# ---------- Horizon Prediction ----------
def get_horizon_prediction(score, pct_change, liq, vol):
    try:
        # Strong momentum
        if score >= 5 and pct_change > 50:
            return "🎯 1H:+10-20% | 4H:+20-50% | 24H:+50-100%"

        # Good momentum
        elif score >= 4 and pct_change > 20:
            return "🎯 1H:+5-15% | 4H:+15-40% | 24H:+30-80%"

        # Moderate momentum
        elif score >= 3:
            return "🎯 1H:+3-10% | 4H:+10-25% | 24H:+20-60%"

        # Weak momentum
        elif score >= 1:
            return "🎯 1H:-5%~+5% | 4H:-10%~+10% | 24H:-20%~+20%"

        # High risk
        else:
            return "⚠️ Uncertain / Rug Risk"

    except:
        return "N/A"

# ---------- parsers ----------
def parse_gmgn(raw):
    if not raw:
        return []
    try:
        j = json.loads(raw)
    except Exception:
        return []
    arr = j.get("data", {}).get("list") or []
    out = []
    for x in arr[:MAX_TOKENS]:
        base = x.get("baseToken") or {}
        out.append({
            "name": base.get("name") or x.get("name"),
            "symbol": base.get("symbol"),
            "address": base.get("address"),
            "price": x.get("price") or base.get("price"),
            "liquidity": x.get("liquidity") or x.get("liquidityUsd"),
            "fdv": x.get("fdv"),
            "volume": x.get("volume24h") or x.get("volume"),
            "price_change": x.get("price_change") or x.get("priceChange")
        })
    return out


def parse_dex(raw):
    if not raw:
        return []
    try:
        j = json.loads(raw)
    except Exception:
        return []
    pairs = j.get("pairs") or []
    out = []
    for p in pairs[:MAX_TOKENS]:
        base = p.get("baseToken") or {}
        price = p.get("priceUsd") or p.get("price")
        liq_obj = p.get("liquidity")
        liq = liq_obj.get("usd") if isinstance(liq_obj, dict) else p.get("liquidity")
        change = None
        for k in ("priceChange", "priceChange24h", "price_change_24h", "priceChange1h"):
            if p.get(k) is not None:
                change = p.get(k)
                break
        out.append({
            "name": base.get("name") or p.get("pairName"),
            "symbol": base.get("symbol"),
            "address": base.get("address") or (p.get("url") or "").split("/")[-1],
            "price": price,
            "liquidity": liq,
            "fdv": p.get("fdv"),
            "volume": (p.get("volume") or {}).get("h24") or p.get("volume"),
            "price_change": change
        })
    return out


def parse_gecko(raw):
    if not raw:
        return []
    try:
        j = json.loads(raw)
    except Exception:
        return []
    arr = j.get("data") or []
    out = []
    for x in arr[:MAX_TOKENS]:
        attrs = x.get("attributes") or {}
        out.append({
            "name": attrs.get("name"),
            "symbol": attrs.get("base_token_symbol"),
            "address": attrs.get("address"),
            "price": attrs.get("price_usd"),
            "liquidity": attrs.get("liquidity_usd"),
            "fdv": attrs.get("fdv_usd"),
            "volume": attrs.get("volume_usd"),
            "price_change": attrs.get("price_change_percentage_24h")
        })
    return out


def parse_pump(raw):
    if not raw:
        return []
    try:
        j = json.loads(raw)
    except Exception:
        return []
    arr = j.get("tokens") or []
    out = []
    for x in arr[:MAX_TOKENS]:
        out.append({
            "name": x.get("name"),
            "symbol": x.get("symbol"),
            "address": x.get("mint"),
            "price": x.get("usd_price"),
            "liquidity": x.get("liquidity_usd"),
            "fdv": x.get("fdv_usd") or x.get("usd_market_cap"),
            "volume": x.get("volume_24h_usd"),
            "price_change": x.get("price_change_24h")
        })
    return out


def scan_and_build_df():
    raw = fetch_url_text(ENDPOINTS["gmgn"])
    tokens = parse_gmgn(raw)
    if not tokens:
        raw = fetch_url_text(ENDPOINTS["dex"])
        tokens = parse_dex(raw)
    if not tokens:
        raw = fetch_url_text(ENDPOINTS["gecko"])
        tokens = parse_gecko(raw)
    if not tokens:
        raw = fetch_url_text(ENDPOINTS["pump"])
        tokens = parse_pump(raw)
    if not tokens:
        print("❌ No tokens from any source.")
        return pd.DataFrame([])

    rows = []
    for t in tokens:
        nm = (t.get("name") or "").upper()
        sym = (t.get("symbol") or "").upper()
        non_solana_keywords = ["USDC", "USDT", "USD", "ETH", "BTC", "BNB", "APT", "ARB", "BASE", "AVAX", "POL", "OP"]
        if any(k in nm or k in sym for k in non_solana_keywords):
            continue
        if "SOL" not in nm and "SOL" not in sym:
            continue
        core_tokens = ["SOL", "JUP", "RAY", "PYTH", "MOBILE", "BONKZ"]
        if sym in core_tokens or nm in core_tokens:
            continue

        addr = t.get("address")
        price = t.get("price")
        liq = t.get("liquidity")
        fdv = t.get("fdv")
        vol = t.get("volume")
        change = t.get("price_change")

        if (price is None or price == "") and addr:
            info = get_price_and_change_from_dexscreener_by_address(addr)
            if info:
                price = info.get("price") or price
                liq = info.get("liquidity") or liq
                vol = info.get("volume") or vol
                change = info.get("price_change") or change

        potential = "💀 RugPull Risk"
        try:
            liq = float(liq) if liq else 0
            vol = float(vol) if vol else 0
            fdv = float(fdv) if fdv else 0
            price = float(price) if price else 0
            pct_change = extract_percent_change_from_obj(change) or 0

            score = 0
            if liq > 200_000: score += 2
            elif liq > 50_000: score += 1
            if vol > 200_000: score += 2
            elif vol > 50_000: score += 1
            if fdv > 500_000 and fdv < 2_000_000_000: score += 1
            if pct_change > 5: score += 1
            elif pct_change < -20: score -= 1
            if price < 0.001: score += 1

            potential = (
                "🚀 100x" if score >= 5 else
                "🔥 10x" if score >= 3 else
                "⚙️ Low" if score >= 1 else
                "💀 RugPull Risk"
            )
            horizon_prediction = get_horizon_prediction(
                score,
                pct_change,
                liq,
                vol
            )
        except:
            pct_change = 0

        pct_display = f"{pct_change:+.2f}%" if change else ""
        spark_html = ""
        if price:
            base = float(price)
            if change:
                history = np.linspace(base * (1 - pct_change / 100 * 0.6), base, num=8)
            else:
                history = [base * 0.99, base]
            spark_html = render_sparkline(history)

        trend_html = (
            "<span style='color:green;'>🟢 Uptrend</span>" if pct_change > 0 else
            "<span style='color:red;'>🔴 Downtrend</span>" if pct_change < 0 else
            "<span style='color:gray;'>⚪ Neutral</span>"
        )

        rows.append({
            "Nama Token": f"{t.get('name') or ''} ({t.get('symbol') or ''})",
            "Market Cap (fdv)": fdv,
            "Address Token": addr,
            "Potential": potential,
            "Price now": price,
            "% Change": pct_display,
            "% Change (num)": pct_change,
            "Horizon Prediction": horizon_prediction,
            "Chart": spark_html,
            "Trend": trend_html
        })
        time.sleep(DELAY)

    df = pd.DataFrame(rows)
    if df.empty:
        print("No valid Solana memecoins found.")
        return df

    df = df.sort_values(by="% Change (num)", ascending=False)

    def fmt_money(x):
        try:
            if not x:
                return ""
            x = float(x)
            if x < 1:
                return f"${x:.8f}".rstrip('0').rstrip('.')
            if x < 1000:
                return f"${x:.2f}"
            if x < 1_000_000:
                return f"${x / 1_000:.1f}K"
            return f"${x / 1_000_000:.2f}M"
        except:
            return x

    def fmt_price(x):
        try:
            if x is None or x == "":
                return ""
            x = float(x)
            if x >= 1:
                return f"${x:,.4f}".rstrip('0').rstrip('.')
            else:
                return f"${x:.10f}".rstrip('0').rstrip('.')
        except:
            return x

    df["Market Cap (fdv)"] = df["Market Cap (fdv)"].apply(fmt_money)
    df["Price now"] = df["Price now"].apply(fmt_price)

    # === ADD MEDAL EMOJI NEXT TO TOKEN NAME (Top 3) ===
    medals = ["🥇", "🥈", "🥉"]
    for i in range(min(3, len(df))):
        df.iloc[i, df.columns.get_loc("Nama Token")] = f"{df.iloc[i]['Nama Token']} {medals[i]}"

    display(HTML("<h3>🪙 Solana Memecoin Scanner — Top 3 Medal + Chart & Trend</h3>"))
    df_html = df[[
        "Nama Token", "Market Cap (fdv)", "Address Token",
        "Potential", "Price now", "% Change", "Horizon Prediction", "Chart", "Trend"
    ]].to_html(escape=False, index=False)
    display(HTML(df_html))
    return df


# === RUN SCANNER ===
df = scan_and_build_df()
